In [1]:
from os.path import commonprefix
import attr
import json
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict

tqdm.pandas(desc="Обработка строк")

In [2]:
@attr.s(frozen=True)
class LemmatizeRule(object):
    cut_prefix = attr.ib(default=0)
    cut_suffix = attr.ib(default=0)
    append_suffix = attr.ib(default='')
    lower = attr.ib(default=False)
    capitalize = attr.ib(default=False)
    upper = attr.ib(default=False)

    def to_json(self):
        return json.dumps(attr.todict(self), ensure_ascii=False)

In [3]:
def predict_lemmatize_rule(word: str, lemma: str):
    def _predict_lemmatize_rule(word: str, lemma: str, **kwargs):
        if len(word) == 0:
            return LemmatizeRule(append_suffix=lemma, **kwargs)

        common_prefix = commonprefix([word, lemma])
        if len(common_prefix) == 0:
            rule = _predict_lemmatize_rule(word[1:], lemma, **kwargs)
            return attr.evolve(rule, cut_prefix=rule.cut_prefix + 1)

        return LemmatizeRule(cut_suffix=len(word) - len(common_prefix),
                                append_suffix=lemma[len(common_prefix):], **kwargs)

    word, lemma = word.replace('ё', 'е'), lemma.replace('ё', 'е')
    return min([
        _predict_lemmatize_rule(word, lemma),
        _predict_lemmatize_rule(word.lower(), lemma, lower=True),
        _predict_lemmatize_rule(word.capitalize(), lemma, capitalize=True),
        _predict_lemmatize_rule(word.upper(), lemma, upper=True)
    ], key=lambda rule: rule.cut_prefix + rule.cut_suffix)

In [4]:
def load_and_dedup_df(p: str|Path):
    df = pd.read_csv(p, sep="\t", index_col=0)
    df = df.drop_duplicates(subset=["form", "pos", "feats", "lemma"])
    df = df[
        ~df["feats"].str.contains("Typo", na=False)
        & ~df["feats"].str.contains("Abbr", na=False)
        & ~df["feats"].str.contains("Anom", na=False)
    ]
    return df

In [5]:
DATADIR = Path("/mnt/data_storage/datasets/generative_lemmatization_datasets/csvs")

In [6]:
df = load_and_dedup_df(DATADIR / "train.csv")

In [7]:
df["lemmatization_rule"] = df.progress_apply(lambda row: predict_lemmatize_rule(row["form"], row["lemma"]), axis=1)

Обработка строк: 100%|██████████| 254267/254267 [00:07<00:00, 32938.27it/s]


In [8]:
df.head()


,form,pos,feats,lemma,subset,filename,lemmatization_rule
0,Снова,ADV,Degree:Pos,снова,rubic_data-master,001_ru_taiga-ud-train.conllu,"LemmatizeRule(cut_prefix=0, cut_suffix=0, appe..."
1,приобрел,VERB,Aspect:Perf Gender:Masc Mood:Ind Number:Sing T...,приобрести,rubic_data-master,001_ru_taiga-ud-train.conllu,"LemmatizeRule(cut_prefix=0, cut_suffix=1, appe..."
2,дозу,NOUN,Animacy:Inan Case:Acc Gender:Fem Number:Sing,доза,rubic_data-master,001_ru_taiga-ud-train.conllu,"LemmatizeRule(cut_prefix=0, cut_suffix=1, appe..."
3,",",PUNCT,NaN,",",rubic_data-master,001_ru_taiga-ud-train.conllu,"LemmatizeRule(cut_prefix=0, cut_suffix=0, appe..."
4,В,ADP,NaN,в,rubic_data-master,001_ru_taiga-ud-train.conllu,"LemmatizeRule(cut_prefix=0, cut_suffix=0, appe..."


In [9]:
unique_rules = df["lemmatization_rule"].unique()
unique_rules.shape

(4762,)

In [10]:
id2rule = {i: rule for i, rule in enumerate(sorted(unique_rules, key=lambda r: (
    r.cut_prefix, r.cut_suffix, r.append_suffix, r.lower, r.capitalize, r.upper
)))}

In [11]:
rule2id = {rule: id for id, rule in id2rule.items()}

In [12]:
df["rule_id"] = df["lemmatization_rule"].map(rule2id)

In [13]:
@attr.s(frozen=True)
class WordForm(object):
    form = attr.ib(default=None)
    lemma=attr.ib(default=None)
    pos=attr.ib(default=None)
    feats=attr.ib(default=None)

    def to_json(self):
        return json.dumps(attr.todict(self), ensure_ascii=False)

    @classmethod
    def from_df_row(cls, df_row):
        return cls(form=df_row["form"], lemma=df_row["lemma"], pos=df_row["pos"], feats=df_row["feats"])

In [15]:
WordForm.from_df_row(df.iloc[1])

WordForm(form='приобрел', lemma='приобрести', pos='VERB', feats='Aspect:Perf Gender:Masc Mood:Ind Number:Sing Tense:Past Transit:Tran VerbForm:Fin Voice:Act')

In [16]:
id2wordforms = defaultdict(set)

for rule_name, rule_group in df.groupby("rule_id"):
    for _, row in rule_group.iterrows():
        wf = WordForm.from_df_row(row)
        id2wordforms[rule_name].add(wf)

In [17]:
len(id2wordforms)

4762

In [18]:
id2wordforms[0]

{WordForm(form='прощупывание', lemma='прощупывание', pos='NOUN', feats='Animacy:Inan Case:Nom Gender:Neut Number:Sing'),
 WordForm(form='полюс', lemma='полюс', pos='NOUN', feats='Animacy:Inan Case:Acc Gender:Masc Number:Sing'),
 WordForm(form='58', lemma='58', pos='NUM', feats='NumForm:Digit NumType:Card'),
 WordForm(form='Маффео', lemma='Маффео', pos='PROPN', feats='Animacy:Anim Case:Gen Gender:Masc InflClass:Ind NameType:Giv Number:Sing'),
 WordForm(form='насаждать', lemma='насаждать', pos='VERB', feats='Aspect:Imp Transit:Tran VerbForm:Inf Voice:Act'),
 WordForm(form='сорт', lemma='сорт', pos='NOUN', feats='Animacy:Inan Case:Acc Gender:Masc Number:Sing'),
 WordForm(form='митинг', lemma='митинг', pos='NOUN', feats='Animacy:Inan Case:Acc Gender:Masc Number:Sing'),
 WordForm(form='дискуссионность', lemma='дискуссионность', pos='NOUN', feats='Animacy:Inan Case:Nom Gender:Fem Number:Sing'),
 WordForm(form='Академинторг', lemma='Академинторг', pos='PROPN', feats='Animacy:Inan Case:Nom Gen

In [17]:

# with open("/mnt/data_storage/datasets/lemmatization_heuristics/lemmatization_rules_train.json", "w") as file:    
#     json.dump(id2rule,file, ensure_ascii=False, indent=2)

In [19]:
df.head()

,form,pos,feats,lemma,subset,filename,lemmatization_rule,rule_id
0,Снова,ADV,Degree:Pos,снова,rubic_data-master,001_ru_taiga-ud-train.conllu,"LemmatizeRule(cut_prefix=0, cut_suffix=0, appe...",3
1,приобрел,VERB,Aspect:Perf Gender:Masc Mood:Ind Number:Sing T...,приобрести,rubic_data-master,001_ru_taiga-ud-train.conllu,"LemmatizeRule(cut_prefix=0, cut_suffix=1, appe...",216
2,дозу,NOUN,Animacy:Inan Case:Acc Gender:Fem Number:Sing,доза,rubic_data-master,001_ru_taiga-ud-train.conllu,"LemmatizeRule(cut_prefix=0, cut_suffix=1, appe...",147
3,",",PUNCT,NaN,",",rubic_data-master,001_ru_taiga-ud-train.conllu,"LemmatizeRule(cut_prefix=0, cut_suffix=0, appe...",0
4,В,ADP,NaN,в,rubic_data-master,001_ru_taiga-ud-train.conllu,"LemmatizeRule(cut_prefix=0, cut_suffix=0, appe...",3


In [20]:
df[df["rule_id"] == 26]

,form,pos,feats,lemma,subset,filename,lemmatization_rule,rule_id
394930,д,ADV,Degree:Cmp,далее,rubic_data-master,SynTagRus_UD2_8.conllu,"LemmatizeRule(cut_prefix=0, cut_suffix=0, appe...",26
